# SMARTRENT: ข้อมูลจริงและการรันโมเดล Phase 2 ต่อ
ข้อมูลที่ผ่านการเตรียม 6,425 รายการเท่านั้น ไม่มีแถวที่ตัดออกหรือข้อมูลรถไฟฟ้าที่ไม่ได้ใช้
Notebook นี้จัดใหม่เพื่อส่งต่อเพื่อน ไม่ใช่ไฟล์บันทึกการรันต้นฉบับ และไม่ใช่ผลทดลอง Phase 3
ชุดทดสอบเดิมถูกดูผลมาแล้ว ใช้เปรียบเทียบย้อนหลัง ไม่ปรับโมเดลไล่ตามคะแนนชุดนี้

## 1. เตรียมโปรแกรม
บนเครื่อง: ติดตั้ง Python 3.12 แล้วรัน `python -m pip install -r requirements.txt` ก่อนเปิด Notebook
บน Google Colab: อัปโหลด ZIP แตกไฟล์ แล้วติดตั้งจาก requirements.txt ก่อนรันเซลล์ถัดไป หากติดตั้งเปลี่ยนเวอร์ชันให้เริ่ม session ใหม่
แพ็กเกจนี้ทดสอบบนเครื่อง ไม่ได้ทดสอบรันบน Colab

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data' / 'train.csv').exists():
    if (ROOT.parent / 'data' / 'train.csv').exists():
        ROOT = ROOT.parent
    else:
        candidates = list(ROOT.glob('*/data/train.csv'))
        if len(candidates) != 1:
            raise FileNotFoundError('Set ROOT to the smartrent-condo-rent-prediction folder')
        ROOT = candidates[0].parents[1]

sys.path.insert(0, str(ROOT / 'python_code'))
from model_pipeline_helpers import load_data, verify_saved, reproduce_ridge, reproduce_all, FEATURES
import pandas as pd

train, validation, test, train_plus_val = load_data(ROOT)
print('Train:', len(train), 'Validation:', len(validation), 'Test:', len(test), 'Train+Val:', len(train_plus_val))

Train: 4217 Validation: 1408 Test: 800 Final refit: 5625


## 2. ข้อมูลที่ใช้ทำนาย
Feature คือข้อมูลที่ให้โมเดลใช้ทำนาย มี 18 ตัวก่อนการแปลงภายในโมเดล Target คือคำตอบที่ต้องการทำนาย ใช้ rental (ค่าเช่าที่ประกาศต่อเดือน หน่วยบาท)
room_id, project_id, condo_link และ size_sqm ใช้ตรวจสอบ/อ่านประกอบ ไม่อยู่ใน FEATURES; size_sqm ถูกใช้ในรูป log_size แล้ว
ค่าว่าง floor_no ตั้งใจเก็บไว้ให้ขั้นตอนในโมเดลเติมจากชุดฝึก อย่าเติมจากข้อมูลรวมก่อนแบ่งชุด

In [2]:
print('Model inputs:',len(FEATURES))
print(FEATURES)
print(train[FEATURES+['rental']].head().to_string(index=False))

Model inputs: 18
['log_size', 'floor_no', 'bedrooms', 'is_duplex', 'n_reported_room_amenities', 'air_conditioner', 'digital_door_lock', 'furnished', 'hot_tub', 'in_room_wifi', 'kitchen_hood', 'kitchen_stove', 'phone', 'refrigerator', 'tv', 'washer', 'water_heater', 'unit_type']
 log_size  floor_no  bedrooms  is_duplex  n_reported_room_amenities  air_conditioner  digital_door_lock  furnished  hot_tub  in_room_wifi  kitchen_hood  kitchen_stove  phone  refrigerator  tv  washer  water_heater unit_type  rental
 3.806662      27.0         1          0                          3                1                  0          1        0             0             0              0      0             0   0       0             1 1 Bedroom   10000
 3.828641       8.0         1          0                          3                1                  0          1        0             0             0              0      0             0   0       0             1 1 Bedroom   11000
 3.828641      33.0      

## 3. ตรวจโมเดลที่ฝึกไว้แล้ว
โหลดโมเดลเดิมและตรวจว่าทำนายตรงผลเดิมครบ 800 รายการ MAE (Mean Absolute Error) คือผิดเฉลี่ยกี่บาท RMSE (Root Mean Squared Error) คือความผิดพลาดที่ให้น้ำหนักกับรายการที่ผิดมาก ยิ่งต่ำยิ่งดี

In [3]:
print(verify_saved(ROOT).to_string(index=False))

      model      MAE_THB     RMSE_THB       R2  MAPE_pct  within20_pct  project_macro_MAE_THB
Saved Ridge 13866.355268 29879.921466 0.708932 34.100326        31.375           20341.924585


## 4. ฝึก Ridge ใหม่ให้เหมือน Phase 2
Ridge Regression คือโมเดลทำนายตัวเลขที่ควบคุมน้ำหนักตัวแปรไม่ให้สูงเกินไป
ฝึกด้วย train เพื่อประเมิน validation แล้วใช้ค่าตั้งเดิมฝึกตัวสุดท้ายด้วย development (train รวม validation) ก่อนทดสอบ test
ผลใหม่เก็บ my_results/ridge ไม่เขียนทับข้อมูลหรือโมเดลต้นฉบับ

In [4]:
print(reproduce_ridge(ROOT,ROOT/'my_results/ridge').to_string(index=False))

     stage model      MAE_THB     RMSE_THB       R2  MAPE_pct  within20_pct  project_macro_MAE_THB
Validation Ridge  9018.112274 14307.842020 0.595859 31.180626     38.210227           15745.943813
      Test Ridge 13866.355268 29879.921466 0.708932 34.100326     31.375000           20341.924585


## 5. ทางเลือก: รันเปรียบเทียบทุกโมเดลเดิม
รันด้วยคำสั่งด้านล่างเมื่ออยากทำซ้ำการเลือกโมเดลทั้งหมด ใช้เวลามากกว่า Ridge อย่างเดียว
Median baseline คือทายค่ามัธยฐานเดียวทุกห้อง; Random Forest คือรวมคำทำนายจากต้นไม้ตัดสินใจหลายต้น; Gradient Boosting คือเพิ่มต้นไม้เพื่อแก้ความผิดพลาดของชุดก่อนหน้า
ขั้นตอนนี้ทำซ้ำการทดลอง Phase 2 ไม่ใช่การปรับปรุง Phase 3
```python
print(reproduce_all(ROOT,ROOT/"my_results/all_models").to_string(index=False))
```

## ข้อควรจำเวลาใช้ต่อ
- ห้ามนำ test ไปฝึกหรือใช้เลือกการตั้งค่า
- ห้ามนำ train+validation ไปวัดซ้ำบน validation หรือรวมกับ test
- preprocessing คือขั้นตอนเติมค่าหาย แปลงประเภทห้อง และปรับขนาดตัวเลข ถูกฝึกจากชุดฝึกอยู่ใน python_code/model_pipeline_helpers.py
- ข้อมูลค่าเช่าปี 2023 ไม่ใช่ราคาตลาดปัจจุบัน
- ไม่มีข้อมูลส่วนกลาง/ทำเลในโมเดลนี้ และไม่ได้แบ่งโครงการเล็กกลางใหญ่
- 0 ของสิ่งอำนวยความสะดวกหมายถึงไม่มีการระบุในประกาศ ไม่ยืนยันว่าไม่มีจริง